In [ ]:
import json
import os

import numpy as np

In [ ]:
SEED_SEQ = np.random.SeedSequence(42)
TRAIN_SEEDS = [int(seed) for seed in SEED_SEQ.generate_state(3)]
EVALUATION_DIR = "../model_evaluation"
OUTPUT_DIR = f"{EVALUATION_DIR}/summary"

CHECKPOINTS = [
    "004000",
    "008000",
    "012000",
    "016000",
    "020000",
    "024000",
    "last",
]
SCENARIOS = [
    "clean",
    "gaussian_noise",
    "blur",
    "brightness_shift",
]
SEVERITIES = {
    "clean": ["default"],
    "gaussian_noise": ["severity_level_1", "severity_level_2", "severity_level_3"],
    "blur": ["severity_level_1", "severity_level_2", "severity_level_3"],
    "brightness_shift": ["severity_level_1", "severity_level_2", "severity_level_3"],
}
EVAL_SEED = 42

In [ ]:
eval_data = {}
allow_missing = False

for scenario in SCENARIOS:
    eval_data[scenario] = {}
    for severity_level in SEVERITIES[scenario]:
        eval_data[scenario][severity_level] = {}
        for training_seed in TRAIN_SEEDS:
            eval_data[scenario][severity_level][training_seed] = {}

            # Get base model
            ckpt = "000000"
            data_path = f"{EVALUATION_DIR}/{scenario}/smolvla_base_{scenario}_{severity_level}_{EVAL_SEED}"
            if allow_missing and not os.path.exists(f"{data_path}/eval_info.json"):
                continue
            eval_data[scenario][severity_level][training_seed][ckpt] = json.load(open(f"{data_path}/eval_info.json", "r"))

            for ckpt in CHECKPOINTS:
                data_path = f"{EVALUATION_DIR}/{scenario}/smolvla_{training_seed}_{ckpt}_{scenario}_{severity_level}_{EVAL_SEED}"
                if allow_missing and not os.path.exists(f"{data_path}/eval_info.json"):
                    continue
                eval_data[scenario][severity_level][training_seed][ckpt] = json.load(open(f"{data_path}/eval_info.json", "r"))

In [ ]:
def compute_success_rate(max_rewards: list[float], successful_condition: float) -> float:

    return sum([r >= successful_condition for r in max_rewards]) / len(max_rewards)

In [ ]:
successful_condition = 0.8

success_rates = {
    scenario: {
        severity_level: {
            training_seed: {
                ckpt: compute_success_rate(
                    eval_data[scenario][severity_level][training_seed][ckpt][
                        "per_task"
                    ][0]["metrics"]["max_rewards"],
                    successful_condition=successful_condition,
                )
                for ckpt in eval_data[scenario][severity_level][training_seed]
            }
            for training_seed in eval_data[scenario][severity_level]
        }
        for severity_level in eval_data[scenario]
    }
    for scenario in eval_data
}

In [ ]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

json.dump(success_rates, open(f"{OUTPUT_DIR}/success_rates.json", "w"), indent=4)